# Summarization con T5 — Teoria e codice commentato

Questo notebook riprende `2_Summarization.ipynb` e lo arricchisce con teoria introduttiva, una roadmap e commenti estesi al codice.

---

## 1. Teoria introduttiva

### 1.1 Cos'è la summarization
Il **riassunto automatico (summarization)** consiste nel produrre un testo più corto che conservi il contenuto informativo essenziale di un testo più lungo. È un compito di tipo **sequence-to-sequence (seq2seq)**: l'input è una sequenza di testo, l'output è un'altra sequenza di testo, generalmente più breve.

Esistono due approcci principali:
- **Extractive**: si selezionano frasi/porzioni già presenti nel testo originale.
- **Abstractive**: il modello *genera* nuove frasi, riformulando i contenuti (è l'approccio usato da T5, quello di questo notebook).

### 1.2 T5: un Transformer encoder-decoder
A differenza di GPT-2 (decoder-only, visto in `1_TextCompletion`), **T5 (Text-to-Text Transfer Transformer)**, di Google, usa l'architettura **Transformer completa**, con:
- un **encoder**, che legge l'intero testo di input e produce una rappresentazione contestuale bidirezionale (può "guardare" sia a sinistra che a destra di ogni token, non essendo autoregressivo);
- un **decoder**, che genera l'output token dopo token, in modo autoregressivo, condizionandosi sia sui token già generati sia sulla rappresentazione prodotta dall'encoder (tramite **cross-attention**).

Questa architettura è quella naturale per compiti in cui input e output sono sequenze testuali diverse tra loro (traduzione, summarization, question answering generativo, ecc.), mentre GPT-2 è più naturale per la pura continuazione di un testo.

### 1.3 L'idea "text-to-text" di T5
L'innovazione principale di T5 è trattare **ogni** task di NLP come un problema testo-in → testo-out, usando un **prefisso testuale** per specificare il compito. Ad esempio:
- `"summarize: <testo>"` → riassunto
- `"translate English to German: <testo>"` → traduzione

Questo è il motivo per cui nel codice vedremo `"summarize: " + text` prima di passare il testo al tokenizer: è così che T5 "sa" quale compito deve svolgere, senza bisogno di un'architettura diversa per ogni task.

### 1.4 Pretraining di T5
T5 è pre-addestrato con un obiettivo chiamato **span corruption** (una variante del *masked language modeling*): porzioni contigue di testo vengono sostituite con token speciali ("sentinel token"), e il modello deve generare in output i frammenti mancanti. Anche in questo caso, come per GPT-2, l'obiettivo è **self-supervised**: non servono etichette umane, solo testo grezzo.

Dopo il pretraining generico, modelli come T5 vengono spesso ulteriormente **fine-tuned** su dataset specifici di summarization (es. CNN/DailyMail). In questo notebook usiamo `t5-small` così come distribuito da Hugging Face: la qualità dei riassunti dipende da quanto bene il pretraining (+ eventuale fine-tuning) copre questo tipo di testo.

## 2. Dove stiamo andando (roadmap del notebook)

1. **Setup ambiente**: installiamo `tf-keras` (dipendenza di `transformers`).
2. **Import**: `torch` e le classi `T5ForConditionalGeneration` / `T5Tokenizer`.
3. **Caricamento di modello e tokenizer pre-addestrati** (`t5-small`).
4. **Definizione della classe `LLMApplications`** con il metodo `summarization`, che aggiunge il prefisso di task, tokenizza, genera e decodifica.
5. **Esecuzione su un primo testo di esempio** (paragrafo sull'AI).
6. **Esecuzione su un secondo testo più tecnico** (descrizione di BERT), per osservare come si comporta il modello su un input più denso e con terminologia più specifica.

### 2.1 Setup dell'ambiente

In [1]:
# Installa tf-keras: dipendenza richiesta da transformers per compatibilità
# con l'API Keras legacy (non useremo direttamente TensorFlow: qui il backend è PyTorch).
! pip install tf-keras


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2.2 Import

In [2]:
# torch: necessario per i tensori restituiti dal tokenizer ('pt') e per
# l'esecuzione del modello T5 (implementazione PyTorch).
import torch

# T5ForConditionalGeneration: T5 con testa per la generazione condizionata
#   (conditional generation = generare una sequenza di output condizionata
#   su una sequenza di input, tramite l'architettura encoder-decoder).
# T5Tokenizer: tokenizer associato a T5 (basato su SentencePiece, non BPE
#   come GPT-2).
from transformers import T5ForConditionalGeneration, T5Tokenizer

### 2.3 Caricamento di modello e tokenizer pre-addestrati

`t5-small` è la versione più piccola della famiglia T5 (circa 60M parametri): scelta per velocità di esecuzione ed occupazione di memoria, a scapito della qualità massima raggiungibile (versioni più grandi come `t5-base`/`t5-large` producono riassunti generalmente migliori).

In [3]:
# Modello per riassunto (T5)
# from_pretrained carica pesi già addestrati: nessun training avviene in
# questo notebook, solo inferenza.
t5_model = T5ForConditionalGeneration.from_pretrained('t5-small')

# Il tokenizer deve corrispondere esattamente al modello caricato, altrimenti
# gli ID dei token non avrebbero lo stesso significato con cui il modello
# è stato addestrato.
t5_tokenizer = T5Tokenizer.from_pretrained('t5-small')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### 2.4 La classe `LLMApplications` con il metodo `summarization`

Stesso pattern del notebook sul text completion: un dizionario `self.applications` che mappa il nome del task alla funzione corrispondente, per rendere la classe estensibile.

In [4]:
class LLMApplications:
    def __init__(self):
        # Inizializza le applicazioni
        self.applications = {
            'summarization': self.summarization
        }

    def summarization(self, text):
        """Genera un riassunto utilizzando T5"""

        # STEP 1 - ENCODE: T5 usa il paradigma "text-to-text", quindi il
        # task va specificato tramite un prefisso testuale ("summarize: ").
        # Senza questo prefisso il modello non saprebbe quale operazione
        # eseguire sul testo in input.
        # truncation=True + max_length=512: se il testo è più lungo di 512
        # token viene troncato, per rispettare il limite di contesto del
        # modello (evita errori a runtime su input troppo lunghi).
        inputs = t5_tokenizer.encode("summarize: " + text, return_tensors='pt', truncation=True, max_length=512)

        # STEP 2 - GENERATE: il decoder di T5 genera il riassunto in modo
        # autoregressivo, condizionandosi sulla rappresentazione prodotta
        # dall'encoder per l'intero input (cross-attention).
        # max_length=150: lunghezza massima (in token) del riassunto generato.
        # num_return_sequences=1: un solo riassunto per input (niente sampling
        # esplicito qui: di default generate() usa greedy decoding, quindi
        # il risultato è deterministico, a differenza del notebook su
        # text completion dove do_sample=True introduceva variabilità).
        outputs = t5_model.generate(inputs, max_length=150, num_return_sequences=1)

        # STEP 3 - DECODE: riconverte gli ID generati in testo leggibile,
        # rimuovendo i token speciali (es. end-of-sequence).
        summary = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
        return summary

### 2.5 Primo esempio: paragrafo generico sull'AI

In [5]:
llm_apps = LLMApplications()

# Summarization
print("SUMMARIZATION:")
long_text = """Artificial intelligence (AI) is transforming industries worldwide. 
From healthcare to finance, AI systems are helping professionals make better decisions, 
automate routine tasks, and discover new insights from data. Machine learning algorithms 
can process vast amounts of information quickly and accurately, leading to improved 
efficiency and innovation across sectors."""

# Chiama il metodo summarization: sotto il cofano esegue encode -> generate -> decode.
summary = llm_apps.summarization(long_text)
print(f"   Original: {long_text}")
print(f"   Summary: {summary}")
print()

SUMMARIZATION:
   Original: Artificial intelligence (AI) is transforming industries worldwide. 
From healthcare to finance, AI systems are helping professionals make better decisions, 
automate routine tasks, and discover new insights from data. Machine learning algorithms 
can process vast amounts of information quickly and accurately, leading to improved 
efficiency and innovation across sectors.
   Summary: AI systems are helping professionals make better decisions, automate routine tasks, and discover new insights from data. AI can process vast amounts of information quickly and accurately, leading to improved efficiency and innovation across sectors.



### 2.6 Secondo esempio: testo tecnico (descrizione di BERT)

Testo più lungo e con terminologia tecnica specifica (masked language modeling, next sentence prediction), utile per osservare i limiti di un modello piccolo come `t5-small` su contenuti densi: il riassunto abstractive può risultare meno fluido o perdere qualche nesso logico rispetto a un modello più grande.

In [6]:
# Summarization
print("SUMMARIZATION:")
long_text = """BERT is a transformers model pretrained on a large corpus of English data in a self-supervised fashion. This means it was pretrained on the raw texts only, with no humans labeling them in any way (which is why it can use lots of publicly available data) with an automatic process to generate inputs and labels from those texts. More precisely, it was pretrained with two objectives:
Masked language modeling (MLM): taking a sentence, the model randomly masks 15% of the words in the input then run the entire masked sentence through the model and has to predict the masked words. This is different from traditional recurrent neural networks (RNNs) that usually see the words one after the other, or from autoregressive models like GPT which internally masks the future tokens. It allows the model to learn a bidirectional representation of the sentence.
Next sentence prediction (NSP): the models concatenates two masked sentences as inputs during pretraining. Sometimes they correspond to sentences that were next to each other in the original text, sometimes not. The model then has to predict if the two sentences were following each other or not.
"""

summary = llm_apps.summarization(long_text)
print(f"   Original: {long_text}")
print(f"   Summary: {summary}")
print()

SUMMARIZATION:
   Original: BERT is a transformers model pretrained on a large corpus of English data in a self-supervised fashion. This means it was pretrained on the raw texts only, with no humans labeling them in any way (which is why it can use lots of publicly available data) with an automatic process to generate inputs and labels from those texts. More precisely, it was pretrained with two objectives:
Masked language modeling (MLM): taking a sentence, the model randomly masks 15% of the words in the input then run the entire masked sentence through the model and has to predict the masked words. This is different from traditional recurrent neural networks (RNNs) that usually see the words one after the other, or from autoregressive models like GPT which internally masks the future tokens. It allows the model to learn a bidirectional representation of the sentence.
Next sentence prediction (NSP): the models concatenates two masked sentences as inputs during pretraining. Sometimes t

## 3. Cosa osservare nei risultati

- A differenza del notebook su GPT-2, qui `generate()` **non** usa `do_sample=True`: di default si comporta in modo (quasi) deterministico, quindi ri-eseguendo la cella con lo stesso input si ottiene lo stesso riassunto.
- Il riassunto **non contiene il testo originale come prefisso** (a differenza del text completion): T5 genera una sequenza di output indipendente, condizionata sull'encoding dell'input tramite cross-attention.
- Il prefisso `"summarize: "` è essenziale: è il modo in cui il paradigma text-to-text di T5 comunica al modello *quale compito* eseguire.
- Con `t5-small` e testi tecnici/densi (es. l'esempio su BERT), il riassunto può risultare più "telegrafico" o perdere alcune relazioni logiche tra le frasi: è un limite legato alla dimensione del modello, non un errore del codice.
- `truncation=True, max_length=512` protegge da input troppo lunghi, ma comporta che eventuale testo oltre i 512 token in ingresso venga semplicemente ignorato dal riassunto.